# Lab 28 - Kaggle GPU Setup (vLLM stable)

Notebook này chạy vLLM trên Kaggle GPU và expose 2 endpoint qua ngrok:

- vLLM OpenAI-compatible API: `/v1/chat/completions`
- Embedding API: `/embed`

Yêu cầu trước khi chạy:

1. Kaggle notebook bật Accelerator = **GPU T4 x2**.
2. Internet bật trong notebook.
3. Thay `YOUR_NGROK_TOKEN` ở cell cấu hình.
4. Chạy cell theo thứ tự từ trên xuống.


## Nguyên nhân bản cũ hay lỗi

- Pin `vllm==0.5.4` quá cũ nhưng lại nâng `transformers`, dễ làm lệch dependency với môi trường Kaggle hiện tại.
- Kaggle image mới có thể có `numpy` 2.x; downgrade `numpy` trong kernel đang chạy có thể gây `numpy.dtype size changed` cho tới khi restart kernel.
- Cài `xformers` rời có thể kéo lại phiên bản `torch`/CUDA không khớp với wheel vLLM.
- Đưa `/usr/local/cuda/lib64/stubs` vào `LD_LIBRARY_PATH` khi chạy runtime có thể làm chương trình bắt nhầm thư viện CUDA stub.
- Cell cũ chờ `/health` vô hạn; nếu process vLLM chết thì notebook chỉ in dấu chấm mãi.
- Không kiểm tra GPU, phiên bản package, log server, ngrok token và smoke test API sau khi server lên.


In [ ]:
# Cell 1 - Install a consistent Kaggle/T4 stack, then restart kernel once
# Run this cell first. If it restarts the kernel, rerun from Cell 1.
from pathlib import Path
import importlib.metadata as md
import os
import subprocess
import sys

MARKER = Path("/kaggle/working/.lab28_stack_installed")
EXPECTED = {
    "numpy": "1.26.4",
    "torch": "2.5.1",
    "torchvision": "0.20.1",
    "torchaudio": "2.5.1",
    "xformers": "0.0.28.post3",
    "vllm": "0.7.3",
    "sentence-transformers": "3.4.1",
    "mlflow-skinny": "2.22.4",
}

def installed_version(package: str):
    try:
        return md.version(package).split("+")[0]
    except md.PackageNotFoundError:
        return None

subprocess.run(["nvidia-smi", "-L"], check=True)
stack_ok = MARKER.exists() and all(installed_version(pkg) == ver for pkg, ver in EXPECTED.items())

if stack_ok:
    print("Serving stack already installed and kernel has been restarted.")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"], check=False)

    packages = [
        "numpy==1.26.4",
        "torch==2.5.1",
        "torchvision==0.20.1",
        "torchaudio==2.5.1",
        "xformers==0.0.28.post3",
        "vllm==0.7.3",
        "transformers>=4.48.2,<4.50",
        "sentence-transformers==3.4.1",
        "fastapi>=0.115,<0.116",
        "uvicorn[standard]>=0.34,<0.35",
        "pyngrok>=7.2,<8",
        "mlflow-skinny==2.22.4",
        "openai>=1.60,<2",
    ]
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall", *packages
    ])

    check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, capture_output=True)
    print(check.stdout or check.stderr or "pip check: no broken requirements found")

    MARKER.write_text("installed\n", encoding="utf-8")
    print("Install complete. Restarting kernel now to clear NumPy/Torch binary state. After restart, rerun from Cell 1.")
    os.kill(os.getpid(), 9)


In [ ]:
# Cell 2 - Preflight checks
import importlib.metadata as md
import os
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("nvidia-smi not found. In Kaggle, set Accelerator to GPU T4 x2 and restart the session.")

import torch
import vllm
import numpy as np
import numpy.random

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("vLLM:", vllm.__version__)
print("Transformers:", md.version("transformers"))
print("xFormers:", md.version("xformers"))
print("MLflow:", md.version("mlflow-skinny"))
print("GPU count:", torch.cuda.device_count())

expected_versions = {
    "numpy": "1.26.4",
    "torch": "2.5.1",
    "torchvision": "0.20.1",
    "torchaudio": "2.5.1",
    "xformers": "0.0.28.post3",
    "vllm": "0.7.3",
    "sentence-transformers": "3.4.1",
}
for package, expected in expected_versions.items():
    installed = md.version(package).split("+")[0]
    if installed != expected:
        raise RuntimeError(f"{package} version mismatch: expected {expected}, got {installed}. Restart runtime and rerun Cell 1.")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Check Kaggle Accelerator settings.")

for idx in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(idx)
    cc = torch.cuda.get_device_capability(idx)
    total_gb = torch.cuda.get_device_properties(idx).total_memory / 1024**3
    print(f"GPU {idx}: {name}, compute capability {cc[0]}.{cc[1]}, {total_gb:.1f} GiB")

if torch.cuda.get_device_capability(0)[0] < 7:
    raise RuntimeError("This notebook expects a T4/P100-class GPU or newer. Use Kaggle GPU T4 x2.")

subprocess.run(["nvidia-smi"], check=False)


In [ ]:
# Cell 3 - Configuration
from pyngrok import ngrok

NGROK_TOKEN = "YOUR_NGROK_TOKEN"  # Replace with your token from https://dashboard.ngrok.com/get-started/your-authtoken

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"
VLLM_PORT = 8001
EMBED_PORT = 8002
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.80

if not NGROK_TOKEN or NGROK_TOKEN == "YOUR_NGROK_TOKEN":
    raise ValueError("Replace NGROK_TOKEN before running this cell.")

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)
print("Configuration OK")


In [ ]:
# Cell 4 - Start vLLM with logs, timeout, health check, and smoke test
import json
import os
import queue
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request

def _clean_ld_library_path(value: str) -> str:
    return ":".join(part for part in value.split(":") if part and "stubs" not in part)

def _http_get_ok(url: str, timeout: int = 3) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            return 200 <= resp.status < 300
    except Exception:
        return False

def _post_json(url: str, payload: dict, timeout: int = 30) -> dict:
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["TOKENIZERS_PARALLELISM"] = "false"
env["VLLM_USE_V1"] = "0"
env["VLLM_ATTENTION_BACKEND"] = "XFORMERS"
env["LD_LIBRARY_PATH"] = _clean_ld_library_path(env.get("LD_LIBRARY_PATH", ""))

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_NAME,
    "--host", "0.0.0.0",
    "--port", str(VLLM_PORT),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--enforce-eager",
    "--trust-remote-code",
]

print("Starting vLLM:", " ".join(cmd))
vllm_process = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

log_queue = queue.Queue()
recent_logs = []

def _reader():
    assert vllm_process.stdout is not None
    for line in vllm_process.stdout:
        log_queue.put(line.rstrip())

threading.Thread(target=_reader, daemon=True).start()

health_url = f"http://127.0.0.1:{VLLM_PORT}/health"
deadline = time.time() + 25 * 60
last_print = 0

while time.time() < deadline:
    while not log_queue.empty():
        line = log_queue.get()
        recent_logs.append(line)
        recent_logs[:] = recent_logs[-80:]
        print(line)

    if vllm_process.poll() is not None:
        raise RuntimeError("vLLM exited before becoming healthy. Last logs:\n" + "\n".join(recent_logs[-40:]))

    if _http_get_ok(health_url):
        break

    if time.time() - last_print > 15:
        print("Waiting for vLLM health check...")
        last_print = time.time()
    time.sleep(2)
else:
    vllm_process.terminate()
    raise TimeoutError("vLLM did not become healthy within 25 minutes. Last logs:\n" + "\n".join(recent_logs[-40:]))

print("vLLM health check OK")

smoke = _post_json(
    f"http://127.0.0.1:{VLLM_PORT}/v1/chat/completions",
    {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Reply with exactly: ok"}],
        "max_tokens": 8,
        "temperature": 0,
    },
    timeout=120,
)
print("Smoke test response:", smoke["choices"][0]["message"]["content"])

tunnel = ngrok.connect(VLLM_PORT, "http")
vllm_url = tunnel.public_url
print(f"VLLM_NGROK_URL={vllm_url}")


In [ ]:
# Cell 5 - Start embedding service
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import torch
import uvicorn

app = FastAPI(title="Lab28 Embedding Service")
EMBED_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cpu"
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=EMBED_DEVICE)
print("Embedding device:", EMBED_DEVICE)

@app.post("/embed")
def embed(data: dict):
    texts = data.get("texts")
    if not isinstance(texts, list) or not texts:
        return {"error": "Request body must contain a non-empty texts list"}
    embeddings = embed_model.encode(texts, normalize_embeddings=True).tolist()
    return {"embeddings": embeddings}

@app.get("/health")
def health():
    return {"status": "ok"}

def run_embed_server():
    uvicorn.run(app, host="0.0.0.0", port=EMBED_PORT, log_level="warning")

threading.Thread(target=run_embed_server, daemon=True).start()

embed_health_url = f"http://127.0.0.1:{EMBED_PORT}/health"
deadline = time.time() + 120
while time.time() < deadline:
    if _http_get_ok(embed_health_url):
        break
    time.sleep(2)
else:
    raise TimeoutError("Embedding service did not become healthy within 120 seconds.")

embed_smoke = _post_json(
    f"http://127.0.0.1:{EMBED_PORT}/embed",
    {"texts": ["hello"]},
    timeout=30,
)
print("Embedding dimension:", len(embed_smoke["embeddings"][0]))

embed_tunnel = ngrok.connect(EMBED_PORT, "http")
embed_url = embed_tunnel.public_url
print(f"EMBED_NGROK_URL={embed_url}")


In [ ]:
# Cell 6 - MLflow logging
import mlflow

mlflow.set_experiment("lab28-integration")

with mlflow.start_run(run_name="vllm-serving-kaggle-stable"):
    mlflow.log_param("model", MODEL_NAME)
    mlflow.log_param("vllm_version", vllm.__version__)
    mlflow.log_param("max_model_len", MAX_MODEL_LEN)
    mlflow.log_metric("gpu_memory_utilization", GPU_MEMORY_UTILIZATION)
    mlflow.set_tag("serving_url", vllm_url)
    mlflow.set_tag("embedding_url", embed_url)
    mlflow.set_tag("status", "production")

print("Integration OK")
print("Use these values in local .env:")
print(f"VLLM_NGROK_URL={vllm_url}")
print(f"EMBED_NGROK_URL={embed_url}")
